In [1]:
import pandas as pd
from skforecast.datasets import fetch_dataset
from darts import TimeSeries
from darts.models import NaiveSeasonal
from darts.metrics import mae, rmse, mape, smape, mase

# 1. Load and prepare data (Darts format)
data = fetch_dataset(name='h2o', raw=True)
data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')
y = TimeSeries.from_dataframe(data, time_col='fecha', value_cols='x', freq='MS')

# 2. Split data into Train and Test sets
# We use the last 24 months for testing (validation)
train, test = y.split_before(len(y) - 24)

# 3. Generate a Forecast
# We use Seasonal Naive as our model
model = NaiveSeasonal(K=12)
model.fit(train)
forecast = model.predict(len(test))

# 4. Calculate Accuracy Metrics
# Note: Compare the 'test' (actual) vs 'forecast' (predicted)

# Scale-Dependent
mae_score = mae(test, forecast)
rmse_score = rmse(test, forecast)

# Percentage (Darts returns 0-100 scale by default)
mape_score = mape(test, forecast)
smape_score = smape(test, forecast)

# Scaled
# MASE needs the training data (insample) to calculate the benchmark error
mase_score = mase(test, forecast, insample=train, m=12)

# 5. Display Results
results_darts = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)', 'sMAPE (%)', 'MASE'],
    'Value': [mae_score, rmse_score, mape_score, smape_score, mase_score]
})

print(results_darts.round(3))

Importing plotly failed. Interactive plots will not work.


╭────────────────────────────────────── h2o ───────────────────────────────────────╮
│ Description:                                                                     │
│ Monthly expenditure ($AUD) on corticosteroid drugs that the Australian health    │
│ system had between 1991 and 2008.                                                │
│                                                                                  │
│ Source:                                                                          │
│ Hyndman R (2023). fpp3: Data for Forecasting: Principles and Practice(3rd        │
│ Edition). http://pkg.robjhyndman.com/fpp3package/,https://github.com/robjhyndman │
│ /fpp3package, http://OTexts.com/fpp3.                                            │
│                                                                                  │
│ URL:                                                                             │
│ https://raw.githubusercontent.com/skforecast/skforecast-                         │
│ datasets/main/data/h2o.csv                                                       │
│                                                                                  │
│ Shape: 204 rows x 2 columns                                                      │
╰──────────────────────────────────────────────────────────────────────────────────╯

      Metric  Value
0        MAE  0.056
1       RMSE  0.075
2   MAPE (%)  6.336
3  sMAPE (%)  6.564
4       MASE  0.929


In [2]:
import numpy as np
import pandas as pd
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.base import ForecastingHorizon
from sktime.performance_metrics.forecasting import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    mean_absolute_scaled_error
)

# data in pandas format for sktime
data = data.set_index('fecha')
y_skt = data['x'].to_period('M') 

# 2. Split data into Train and Test sets
# We use the last 24 months for testing
y_train, y_test = temporal_train_test_split(y_skt, test_size=24)

# 3. Generate a Forecast
# NaiveForecaster with strategy="last" and sp=12 is equivalent to Seasonal Naive
forecaster = NaiveForecaster(strategy="last", sp=12)
forecaster.fit(y_train)
# Predict test set
fh = ForecastingHorizon(np.arange(1, len(y_test)+1), is_relative=True)
y_pred = forecaster.predict(fh)

# 4. Calculate Accuracy Metrics

# Scale-Dependent
mae_score = mean_absolute_error(y_test, y_pred)
# sktime uses mean_squared_error with square_root=True for RMSE
rmse_score = mean_squared_error(y_test, y_pred, square_root=True)

# Percentage
# Note: sktime returns ratios (0-1 scale) or 0-2 scale for sMAPE. 
# We multiply by 100 to get percentages (0-100% or 0-200%)
mape_score = mean_absolute_percentage_error(y_test, y_pred, symmetric=False) * 100
smape_score = mean_absolute_percentage_error(y_test, y_pred, symmetric=True) * 100

# Scaled (MASE)
# We must pass y_train to calculate the benchmark scaling.
mase_score = mean_absolute_scaled_error(y_test, y_pred, y_train=y_train, sp=12)

# 5. Display Results
results_sktime = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)', 'sMAPE (%)', 'MASE'],
    'Value': [mae_score, rmse_score, mape_score, smape_score, mase_score]
})

print(results_sktime.round(3))

      Metric  Value
0        MAE  0.056
1       RMSE  0.075
2   MAPE (%)  6.336
3  sMAPE (%)  6.564
4       MASE  0.929


/home/alumno/Desktop/datos/Time Series/.venv/lib/python3.9/site-packages/skbase/base/_base.py:1342: FutureWarning: tag 'handles-missing-data' will be removed in sktime version 1.0.0 and replaced by 'capability:missing_values', please use 'capability:missing_values' instead
  self._deprecate_tag_warn(collected_tags)
